# Churn Prediction for a Telecommunications Provider

You are a data analyst in a telecommunications company. Your company is facing a high churn rate and you are tasked with creating a model to predict which customers are most likely to churn next. 

In [1]:
# Define your imports here
import numpy as np
import pandas as pd

# We will use the balanced accuracy score to evaluate our models
from sklearn.metrics import balanced_accuracy_score

## Load data

In [2]:
customers = pd.read_csv("customers.csv")
payment_info = pd.read_csv("payment_info.csv")
service_options = pd.read_csv("service_options.csv")
churn = pd.read_csv("churn_analysis.csv")

In [3]:
def one_hot_encoding(df: pd.DataFrame) -> pd.DataFrame:
    """ A function to automatically apply one-hot encoding to all string and categorical variables in a dataframe df, excluding the customer_id column. 
    
    Example usage: df_encoded = one_hot_encoding(df)
    
    """
    
    cat_cols = df.select_dtypes(include=["object", "category"]).columns
    cat_cols = cat_cols.drop("customer_id", errors="ignore")

    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

    # convert True/False → 1/0
    bool_cols = df.select_dtypes(include="bool").columns
    df[bool_cols] = df[bool_cols].astype(int)

    return df


## Description of the available data:

*customers.csv*: Customer profile and service subscription information
- customer_id: Unique identifier for each customer
- gender: Customer’s gender
- age: Customer’s age (in years)
- partner: Indicates whether the customer has a partner (Yes/No)
- number_of_dependents: Number of dependents associated with the customer
- married: Indicates whether the customer is married (Yes/No)
- online_security: Subscription to an online security add-on (Yes/No)
- online_backup: Subscription to an online backup add-on (Yes/No)
- device_protection: Subscription to a device protection add-on (Yes/No)
- premium_tech_support: Subscription to premium technical support (Yes/No)
- streaming_tv: Subscription to streaming TV services (Yes/No)
- streaming_movies: Subscription to streaming movie services (Yes/No)
- streaming_music: Subscription to streaming music services (Yes/No)
- internet_type: Type of internet service (e.g., Fiber Optic, DSL, Cable)

*payment_info.csv*: Customer payment, billing, and revenue information
- customer_id: Unique identifier for each customer
- contract: Type of customer contract
- paperless_billing: Indicates whether the customer uses paperless billing (Yes/No)
- payment_method: Method used for payment
- monthly_charges: Recurring monthly charges
- avg_monthly_long_distance_charges: Average monthly long-distance charges
- total_charges: Total charges incurred by the customer
- total_refunds: Total amount refunded to the customer
- total_extra_data_charges: Total charges for extra data usage
- total_long_distance_charges: Total long-distance charges
- total_revenue: Total revenue generated by the customer
- unit: Unit in which the revenues are expressed (e.g., cents, dollars)


*service_options.csv*: Customer service usage, tenure, and marketing information
- customer_id: Unique identifier for each customer
- tenure: Customer tenure (number of months with the company)
- internet_service: Indicates whether the customer has internet service (Yes/No)
- phone_service: Indicates whether the customer has phone service (Yes/No)
- multiple_lines: Indicates whether the customer has multiple service lines (Yes/No)
- avg_monthly_gb_download: Average monthly data download in gigabytes (GB)
- unlimited_data: Indicates whether the customer has an unlimited data plan (Yes/No)
- offer: Most recent marketing offer accepted by the customer (None, Offer A–E)
- referred_a_friend: Indicates whether the customer referred a friend (Yes/No)
- number_of_referrals: Number of referrals made by the customer

*churn.csv*: Customer churn outcome information
- customer_id: Unique identifier for each customer
- churn: Indicates whether the customer churned in the following quarter (Yes/No)



__Task__: Some customer_ids do not have a churn label. Use the given data sets to develop a predictive model to predict the churn of exactly these customers. Remember, churn is binary.  

# Data Preparation

## Data Cleaning

In [4]:
customers.head()

,customer_id,gender,age,partner,number_of_dependents,married,online_security,online_backup,device_protection,premium_tech_support,streaming_tv,streaming_movies,streaming_music,internet_type
0,0002-ORFBO,Female,37,Yes,0,Yes,No,Yes,No,Yes,Yes,No,No,cable
1,0003-MKNFE,m,46,No,0,No,No,No,No,No,No,Yes,Yes,cable
2,0004-TLHLJ,Male,50,No,0,NaN,No,No,Yes,No,No,No,No,Fiber
3,0011-IGKFF,Male,78,Yes,0,Yes,No,Yes,Yes,No,Yes,Yes,No,Fiber
4,0013-EXCHZ,Female,75,Yes,0,Yes,No,No,No,Yes,Yes,No,No,Fiber Optic


In [5]:
customers.dtypes

customer_id             object
gender                  object
age                      int64
partner                 object
number_of_dependents     int64
married                 object
online_security         object
online_backup           object
device_protection       object
premium_tech_support    object
streaming_tv            object
streaming_movies        object
streaming_music         object
internet_type           object
dtype: object

In [6]:
customers['gender'].value_counts()

gender
Male      907
m         905
M         887
f         881
F         871
Female    871
female    865
male      856
Name: count, dtype: int64

In [7]:
customers['gender'] = customers['gender'].map({
    'Male': 'Male',
    'm': 'Male',
    'M': 'Male',
    'f': 'Female',
    'F': 'Female',
    'male': 'Male'
}).astype('category')

In [8]:
one_hot_encoding(customers)

,customer_id,age,number_of_dependents,gender_Male,partner_Yes,married_Yes,online_security_Yes,online_backup_Yes,device_protection_Yes,premium_tech_support_Yes,streaming_tv_Yes,streaming_movies_Yes,streaming_music_Yes,internet_type_DSL,internet_type_DSL,internet_type_Fiber,internet_type_Fiber Optic,internet_type_Optic,internet_type_cable
0,0002-ORFBO,37,0,0,1,1,0,1,0,1,1,0,0,0,0,0,0,0,1
1,0003-MKNFE,46,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1
2,0004-TLHLJ,50,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0
3,0011-IGKFF,78,0,1,1,1,0,1,1,0,1,1,0,0,0,1,0,0,0
4,0013-EXCHZ,75,0,0,1,1,0,0,0,1,1,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,9987-LUTYD,20,0,0,0,0,1,0,0,1,0,0,1,0,1,0,0,0,0
7039,9992-RRAMN,40,0,1,1,1,0,0,0,0,0,1,1,0,0,1,0,0,0
7040,9992-UJOEL,22,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0
7041,9993-LHIEB,21,0,1,1,0,1,0,1,1,0,1,1,0,0,0,0,0,1


In [9]:
customers.dtypes

customer_id               object
gender                  category
age                        int64
partner                   object
number_of_dependents       int64
married                   object
online_security           object
online_backup             object
device_protection         object
premium_tech_support      object
streaming_tv              object
streaming_movies          object
streaming_music           object
internet_type             object
dtype: object

In [10]:
payment_info.head()

,customer_id,contract,paperless_billing,payment_method,monthly_ charges,avg_monthly_long_distance_charges,total_charges,total_refunds,total_extra_data_charges,total_long_distance_charges,total_revenue,unit
0,0002-ORFBO,One Year,Yes,Mailed check,65.6,42.39,593.30,0.00,0,381.51,974.81,dollars
1,0003-MKNFE,Month-to-Month,No,Mailed check,5990.0,10.69,542.40,38.33,10,96.21,610.28,cents
2,0004-TLHLJ,Month-to-Month,Yes,Electronic check,7390.0,33.65,280.85,0.00,0,134.60,415.45,cents
3,0011-IGKFF,Month-to-Month,Yes,Electronic check,98.0,27.82,1237.85,0.00,0,361.66,1599.51,dollars
4,0013-EXCHZ,Month-to-Month,Yes,Mailed check,83.9,7.38,267.40,0.00,0,22.14,289.54,dollars


In [11]:
payment_info.rename(columns={'monthly_ charges': 'monthly_charges'}, inplace=True)
payment_info.dtypes

customer_id                           object
contract                              object
paperless_billing                     object
payment_method                        object
monthly_charges                      float64
avg_monthly_long_distance_charges    float64
total_charges                        float64
total_refunds                        float64
total_extra_data_charges               int64
total_long_distance_charges          float64
total_revenue                        float64
unit                                  object
dtype: object

In [12]:
payment_info['unit'].value_counts()

unit
dollars    4733
cents      2310
Name: count, dtype: int64

In [13]:
payment_info.loc[payment_info['unit'] == 'cent', 'monthly_charges'] = payment_info.loc[payment_info['unit'] == 'cent', 'monthly_charges']*100
payment_info.loc[payment_info['unit'] == 'cent', 'avg_monthly_long_distance_charges'] = payment_info.loc[payment_info['unit'] == 'cent', 'avg_monthly_long_distance_charges']*100
payment_info.loc[payment_info['unit'] == 'cent', 'total_charges'] = payment_info.loc[payment_info['unit'] == 'cent', 'total_charges']*100
payment_info.loc[payment_info['unit'] == 'cent', 'total_refunds'] = payment_info.loc[payment_info['unit'] == 'cent', 'total_refunds']*100
payment_info.loc[payment_info['unit'] == 'cent', 'total_extra_data_charges'] = payment_info.loc[payment_info['unit'] == 'cent', 'total_extra_data_charges']*100
payment_info.loc[payment_info['unit'] == 'cent', 'total_long_distance_charges']= payment_info.loc[payment_info['unit'] == 'cent', 'total_long_distance_charges']*100
payment_info.loc[payment_info['unit'] == 'cent', 'total_revenue']= payment_info.loc[payment_info['unit'] == 'cent', 'total_revenue']*100

In [14]:
payment_info.head()

,customer_id,contract,paperless_billing,payment_method,monthly_charges,avg_monthly_long_distance_charges,total_charges,total_refunds,total_extra_data_charges,total_long_distance_charges,total_revenue,unit
0,0002-ORFBO,One Year,Yes,Mailed check,65.6,42.39,593.30,0.00,0,381.51,974.81,dollars
1,0003-MKNFE,Month-to-Month,No,Mailed check,5990.0,10.69,542.40,38.33,10,96.21,610.28,cents
2,0004-TLHLJ,Month-to-Month,Yes,Electronic check,7390.0,33.65,280.85,0.00,0,134.60,415.45,cents
3,0011-IGKFF,Month-to-Month,Yes,Electronic check,98.0,27.82,1237.85,0.00,0,361.66,1599.51,dollars
4,0013-EXCHZ,Month-to-Month,Yes,Mailed check,83.9,7.38,267.40,0.00,0,22.14,289.54,dollars


In [15]:
payment_info = one_hot_encoding(payment_info)

In [16]:
payment_info.dtypes

customer_id                                object
monthly_charges                           float64
avg_monthly_long_distance_charges         float64
total_charges                             float64
total_refunds                             float64
total_extra_data_charges                    int64
total_long_distance_charges               float64
total_revenue                             float64
contract_One Year                           int64
contract_Two Year                           int64
paperless_billing_Yes                       int64
payment_method_Credit card (automatic)      int64
payment_method_Electronic check             int64
payment_method_Mailed check                 int64
unit_dollars                                int64
dtype: object

In [17]:
churn.dtypes

customer_id     object
churn          float64
dtype: object

## Merge datasets

In [18]:
customers_with_payment_info = pd.merge(customers, payment_info, how='inner')

In [19]:
data = pd.merge(customers_with_payment_info, churn, how='inner')

In [20]:
data.head()

,customer_id,gender,age,partner,number_of_dependents,married,online_security,online_backup,device_protection,premium_tech_support,...,total_long_distance_charges,total_revenue,contract_One Year,contract_Two Year,paperless_billing_Yes,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,unit_dollars,churn
0,0002-ORFBO,NaN,37,Yes,0,Yes,No,Yes,No,Yes,...,381.51,974.81,1,0,1,0,0,1,1,NaN
1,0003-MKNFE,Male,46,No,0,No,No,No,No,No,...,96.21,610.28,0,0,0,0,0,1,0,0.0
2,0004-TLHLJ,Male,50,No,0,NaN,No,No,Yes,No,...,134.60,415.45,0,0,1,0,1,0,0,NaN
3,0011-IGKFF,Male,78,Yes,0,Yes,No,Yes,Yes,No,...,361.66,1599.51,0,0,1,0,1,0,1,1.0
4,0013-EXCHZ,NaN,75,Yes,0,Yes,No,No,No,Yes,...,22.14,289.54,0,0,1,0,0,1,1,1.0


In [21]:
married_mode = data['married'].mode()[0]
data['married'] = data['married'].fillna(married_mode)

In [22]:
data.head()

,customer_id,gender,age,partner,number_of_dependents,married,online_security,online_backup,device_protection,premium_tech_support,...,total_long_distance_charges,total_revenue,contract_One Year,contract_Two Year,paperless_billing_Yes,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,unit_dollars,churn
0,0002-ORFBO,NaN,37,Yes,0,Yes,No,Yes,No,Yes,...,381.51,974.81,1,0,1,0,0,1,1,NaN
1,0003-MKNFE,Male,46,No,0,No,No,No,No,No,...,96.21,610.28,0,0,0,0,0,1,0,0.0
2,0004-TLHLJ,Male,50,No,0,No,No,No,Yes,No,...,134.60,415.45,0,0,1,0,1,0,0,NaN
3,0011-IGKFF,Male,78,Yes,0,Yes,No,Yes,Yes,No,...,361.66,1599.51,0,0,1,0,1,0,1,1.0
4,0013-EXCHZ,NaN,75,Yes,0,Yes,No,No,No,Yes,...,22.14,289.54,0,0,1,0,0,1,1,1.0


In [23]:
print(data.isnull().any())

customer_id                               False
gender                                     True
age                                       False
partner                                    True
number_of_dependents                      False
married                                   False
online_security                           False
online_backup                             False
device_protection                         False
premium_tech_support                      False
streaming_tv                              False
streaming_movies                          False
streaming_music                           False
internet_type                              True
monthly_charges                           False
avg_monthly_long_distance_charges         False
total_charges                             False
total_refunds                             False
total_extra_data_charges                  False
total_long_distance_charges               False
total_revenue                           

In [24]:
internet_type_mode = data['internet_type'].mode()[0]
data['internet_type'] = data['internet_type'].fillna(internet_type_mode)

In [25]:
print(data.isnull().any())

customer_id                               False
gender                                     True
age                                       False
partner                                    True
number_of_dependents                      False
married                                   False
online_security                           False
online_backup                             False
device_protection                         False
premium_tech_support                      False
streaming_tv                              False
streaming_movies                          False
streaming_music                           False
internet_type                             False
monthly_charges                           False
avg_monthly_long_distance_charges         False
total_charges                             False
total_refunds                             False
total_extra_data_charges                  False
total_long_distance_charges               False
total_revenue                           

In [26]:
gender_mode = data['gender'].mode()[0]
data['gender'] = data['gender'].fillna(gender_mode)

partner_mode = data['partner'].mode()[0]
data['partner'] = data['partner'].fillna(partner_mode)
print(data.isnull().any())

customer_id                               False
gender                                    False
age                                       False
partner                                   False
number_of_dependents                      False
married                                   False
online_security                           False
online_backup                             False
device_protection                         False
premium_tech_support                      False
streaming_tv                              False
streaming_movies                          False
streaming_music                           False
internet_type                             False
monthly_charges                           False
avg_monthly_long_distance_charges         False
total_charges                             False
total_refunds                             False
total_extra_data_charges                  False
total_long_distance_charges               False
total_revenue                           

# Modelling 

## Train / Test Split (optional)

In [27]:
# Split data into labeled (train) and unlabeled (pred) sets
data_train = data[data['churn'].notna()].copy()
data_pred = data[data['churn'].isna()].copy()

print(f"Training Data Shape: {data_train.shape}")
print(f"Prediction Data Shape: {data_pred.shape}")

Training Data Shape: (5282, 29)
Prediction Data Shape: (1761, 29)


## Model Fitting

In [28]:
data_train.rename(columns={
    'contract_One Year': 'contract_One_Year',
    'contract_Two Year': 'contract_Two_Year',
    'payment_method_Credit card (automatic)': 'payment_method_Credit_card_automatic',
    'payment_method_Electronic check': 'payment_method_Electronic_check',
    'payment_method_Mailed check': 'payment_method_Mailed_check'
}, inplace=True)

In [29]:
data_pred.rename(columns={
    'contract_One Year': 'contract_One_Year',
    'contract_Two Year': 'contract_Two_Year',
    'payment_method_Credit card (automatic)': 'payment_method_Credit_card_automatic',
    'payment_method_Electronic check': 'payment_method_Electronic_check',
    'payment_method_Mailed check': 'payment_method_Mailed_check'
}, inplace=True)

In [30]:
data_train.columns

Index(['customer_id', 'gender', 'age', 'partner', 'number_of_dependents',
       'married', 'online_security', 'online_backup', 'device_protection',
       'premium_tech_support', 'streaming_tv', 'streaming_movies',
       'streaming_music', 'internet_type', 'monthly_charges',
       'avg_monthly_long_distance_charges', 'total_charges', 'total_refunds',
       'total_extra_data_charges', 'total_long_distance_charges',
       'total_revenue', 'contract_One_Year', 'contract_Two_Year',
       'paperless_billing_Yes', 'payment_method_Credit_card_automatic',
       'payment_method_Electronic_check', 'payment_method_Mailed_check',
       'unit_dollars', 'churn'],
      dtype='object')

In [31]:
data_train.rename(columns={
    'payment_method_Credit_card (automatic)': 'payment_method_Credit_card_automatic',
}, inplace=True)

In [32]:
data_pred.rename(columns={
    'payment_method_Credit_card (automatic)': 'payment_method_Credit_card_automatic',
}, inplace=True)

In [33]:
data_train.columns

Index(['customer_id', 'gender', 'age', 'partner', 'number_of_dependents',
       'married', 'online_security', 'online_backup', 'device_protection',
       'premium_tech_support', 'streaming_tv', 'streaming_movies',
       'streaming_music', 'internet_type', 'monthly_charges',
       'avg_monthly_long_distance_charges', 'total_charges', 'total_refunds',
       'total_extra_data_charges', 'total_long_distance_charges',
       'total_revenue', 'contract_One_Year', 'contract_Two_Year',
       'paperless_billing_Yes', 'payment_method_Credit_card_automatic',
       'payment_method_Electronic_check', 'payment_method_Mailed_check',
       'unit_dollars', 'churn'],
      dtype='object')

In [34]:
data_train.rename(columns={
    'payment_method_Credit card (automatic)': 'payment_method_Credit_card_automatic',
}, inplace=True)
data_train.columns

Index(['customer_id', 'gender', 'age', 'partner', 'number_of_dependents',
       'married', 'online_security', 'online_backup', 'device_protection',
       'premium_tech_support', 'streaming_tv', 'streaming_movies',
       'streaming_music', 'internet_type', 'monthly_charges',
       'avg_monthly_long_distance_charges', 'total_charges', 'total_refunds',
       'total_extra_data_charges', 'total_long_distance_charges',
       'total_revenue', 'contract_One_Year', 'contract_Two_Year',
       'paperless_billing_Yes', 'payment_method_Credit_card_automatic',
       'payment_method_Electronic_check', 'payment_method_Mailed_check',
       'unit_dollars', 'churn'],
      dtype='object')

In [35]:
data_pred.rename(columns={
    'payment_method_Credit card (automatic)': 'payment_method_Credit_card_automatic',
}, inplace=True)
data_pred.columns

Index(['customer_id', 'gender', 'age', 'partner', 'number_of_dependents',
       'married', 'online_security', 'online_backup', 'device_protection',
       'premium_tech_support', 'streaming_tv', 'streaming_movies',
       'streaming_music', 'internet_type', 'monthly_charges',
       'avg_monthly_long_distance_charges', 'total_charges', 'total_refunds',
       'total_extra_data_charges', 'total_long_distance_charges',
       'total_revenue', 'contract_One_Year', 'contract_Two_Year',
       'paperless_billing_Yes', 'payment_method_Credit_card_automatic',
       'payment_method_Electronic_check', 'payment_method_Mailed_check',
       'unit_dollars', 'churn'],
      dtype='object')

In [36]:
print(data_train.isnull().sum())

customer_id                             0
gender                                  0
age                                     0
partner                                 0
number_of_dependents                    0
married                                 0
online_security                         0
online_backup                           0
device_protection                       0
premium_tech_support                    0
streaming_tv                            0
streaming_movies                        0
streaming_music                         0
internet_type                           0
monthly_charges                         0
avg_monthly_long_distance_charges       0
total_charges                           0
total_refunds                           0
total_extra_data_charges                0
total_long_distance_charges             0
total_revenue                           0
contract_One_Year                       0
contract_Two_Year                       0
paperless_billing_Yes             

In [37]:
data_train.dtypes

customer_id                               object
gender                                  category
age                                        int64
partner                                   object
number_of_dependents                       int64
married                                   object
online_security                           object
online_backup                             object
device_protection                         object
premium_tech_support                      object
streaming_tv                              object
streaming_movies                          object
streaming_music                           object
internet_type                             object
monthly_charges                          float64
avg_monthly_long_distance_charges        float64
total_charges                            float64
total_refunds                            float64
total_extra_data_charges                   int64
total_long_distance_charges              float64
total_revenue       

In [38]:
import statsmodels.formula.api as smf
formula = """churn ~ gender + age + partner + number_of_dependents + married + online_security + online_backup + device_protection + premium_tech_support + streaming_tv + streaming_movies + streaming_music + internet_type + monthly_charges + total_charges + total_refunds + total_extra_data_charges + total_long_distance_charges + total_revenue + contract_One_Year + contract_Two_Year + paperless_billing_Yes + payment_method_Credit_card_automatic + payment_method_Electronic_check + payment_method_Mailed_check"""

model = smf.logit(formula, data=data_train)
results = model.fit()
print(results.summary())

         Current function value: 0.394865
         Iterations: 35
                           Logit Regression Results                           
Dep. Variable:                  churn   No. Observations:                 5282
Model:                          Logit   Df Residuals:                     5251
Method:                           MLE   Df Model:                           30
Date:                Fri, 13 Mar 2026   Pseudo R-squ.:                  0.3146
Time:                        16:42:39   Log-Likelihood:                -2085.7
converged:                      False   LL-Null:                       -3043.2
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------
Intercept                               -1.0083      0.238     -4.237      0.000      -1.475

/home/zishan/baml-venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [39]:
# Questionable: ['gender', 'partner', 'married', 'online_backup', 'device_protection', 'streaming_movies', 'streaming_music', 'internet_type', 'monthly_charges', 'total_charges', 'total_refunds', 'total_extra_data_charges', 'total_long_distance_charges', 'total_revenue', 'payment_method_Credit_card_automatic']
questionable = ['gender', 'partner', 'married', 'online_backup', 'device_protection', 'streaming_movies', 'streaming_music', 'internet_type', 'monthly_charges', 'total_charges', 'total_refunds', 'total_extra_data_charges', 'total_long_distance_charges', 'total_revenue', 'payment_method_Credit_card_automatic']
data_train.drop(columns=questionable, inplace=True)

In [40]:
data_train.columns

Index(['customer_id', 'age', 'number_of_dependents', 'online_security',
       'premium_tech_support', 'streaming_tv',
       'avg_monthly_long_distance_charges', 'contract_One_Year',
       'contract_Two_Year', 'paperless_billing_Yes',
       'payment_method_Electronic_check', 'payment_method_Mailed_check',
       'unit_dollars', 'churn'],
      dtype='object')

In [41]:
formula = """churn ~ age + number_of_dependents + online_security + premium_tech_support + streaming_tv + contract_One_Year + contract_Two_Year + paperless_billing_Yes + payment_method_Electronic_check + payment_method_Mailed_check"""

model = smf.logit(formula, data=data_train)
results = model.fit()
print(results.summary())

Optimization terminated successfully.
         Current function value: 0.405242
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:                  churn   No. Observations:                 5282
Model:                          Logit   Df Residuals:                     5271
Method:                           MLE   Df Model:                           10
Date:                Fri, 13 Mar 2026   Pseudo R-squ.:                  0.2966
Time:                        16:42:42   Log-Likelihood:                -2140.5
converged:                       True   LL-Null:                       -3043.2
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept                          -1.2790      0.144     -8.881  

In [42]:
data_train.columns

Index(['customer_id', 'age', 'number_of_dependents', 'online_security',
       'premium_tech_support', 'streaming_tv',
       'avg_monthly_long_distance_charges', 'contract_One_Year',
       'contract_Two_Year', 'paperless_billing_Yes',
       'payment_method_Electronic_check', 'payment_method_Mailed_check',
       'unit_dollars', 'churn'],
      dtype='object')

In [43]:
data_train.drop(columns=['avg_monthly_long_distance_charges'], inplace=True)

In [44]:
formula = """churn ~ age + number_of_dependents + online_security + premium_tech_support + streaming_tv + contract_One_Year + contract_Two_Year + paperless_billing_Yes + payment_method_Electronic_check + payment_method_Mailed_check"""

model = smf.logit(formula, data=data_train)
results = model.fit()
print(results.summary())

Optimization terminated successfully.
         Current function value: 0.405242
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:                  churn   No. Observations:                 5282
Model:                          Logit   Df Residuals:                     5271
Method:                           MLE   Df Model:                           10
Date:                Fri, 13 Mar 2026   Pseudo R-squ.:                  0.2966
Time:                        16:42:44   Log-Likelihood:                -2140.5
converged:                       True   LL-Null:                       -3043.2
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept                          -1.2790      0.144     -8.881  

In [45]:
data_pred.columns

Index(['customer_id', 'gender', 'age', 'partner', 'number_of_dependents',
       'married', 'online_security', 'online_backup', 'device_protection',
       'premium_tech_support', 'streaming_tv', 'streaming_movies',
       'streaming_music', 'internet_type', 'monthly_charges',
       'avg_monthly_long_distance_charges', 'total_charges', 'total_refunds',
       'total_extra_data_charges', 'total_long_distance_charges',
       'total_revenue', 'contract_One_Year', 'contract_Two_Year',
       'paperless_billing_Yes', 'payment_method_Credit_card_automatic',
       'payment_method_Electronic_check', 'payment_method_Mailed_check',
       'unit_dollars', 'churn'],
      dtype='object')

In [46]:
data_pred.columns

Index(['customer_id', 'gender', 'age', 'partner', 'number_of_dependents',
       'married', 'online_security', 'online_backup', 'device_protection',
       'premium_tech_support', 'streaming_tv', 'streaming_movies',
       'streaming_music', 'internet_type', 'monthly_charges',
       'avg_monthly_long_distance_charges', 'total_charges', 'total_refunds',
       'total_extra_data_charges', 'total_long_distance_charges',
       'total_revenue', 'contract_One_Year', 'contract_Two_Year',
       'paperless_billing_Yes', 'payment_method_Credit_card_automatic',
       'payment_method_Electronic_check', 'payment_method_Mailed_check',
       'unit_dollars', 'churn'],
      dtype='object')

## Prediction

In [47]:
data_pred['prediction'] = (results.predict(data_pred)> 0.5).astype(int)

Create the final submission by generating a csv file with exactly the following format:

```csv
id,prediction
0002-ORFBO,0
0004-TLHLJ,0
0014-BMAQU,0
0023-UYUPN,0
0023-XUOPT,0
0027-KWYKW,0
0031-PVLZI,0
0042-RLHYP,0
...
```

The id column contains the customer_id, and the prediction column contains the predicted label.

In [48]:
sample_prediction = pd.read_csv("sample_prediction.csv")
sample_prediction.head()

,id,prediction
0,0002-ORFBO,0
1,0004-TLHLJ,0
2,0014-BMAQU,0
3,0023-UYUPN,0
4,0023-XUOPT,0


In [49]:
y_train = data_train['churn']
y_train_pred = (results.predict(data_train)> 0.5).astype(int)

y_test = sample_prediction['prediction']
y_test_pred = (results.predict(data_pred)> 0.5).astype(int)

In [50]:
train_bal_acc = balanced_accuracy_score(y_train, y_train_pred)
test_bal_acc = balanced_accuracy_score(y_test, y_test_pred)
print("Train balanced accuracy:", train_bal_acc)
print("Test balanced accuracy :", test_bal_acc)

Train balanced accuracy: 0.7383265490828548
Test balanced accuracy : 0.7700170357751278


/home/zishan/baml-venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [51]:
bal_acc = (train_bal_acc + test_bal_acc)/2
print(bal_acc)

0.7541717924289912


In [52]:
data_pred['prediction'] = (results.predict(data_pred)> 0.5).astype(int)

In [53]:
data_pred[['customer_id', 'prediction']].to_csv('predictions.csv', index=False)